# Resolved species taxonomy -> `c14_master_v08.xlsx` mapping

Self-contained continuation of `species_split_for_study_etl.ipynb`'s steps 1-5: takes a
hand-completed species resolution list (`data/manual_species_resolved_test.csv` - every row has
`resolved_order`/`resolved_family`/`resolved_genus`/`resolved_species`/`common_name_text` filled
in, including rows the automated pass originally left blank), resolves real-or-new SEAD ids for
every one of them, then maps that resolved taxonomy back onto `c14_master_v08.xlsx` at the
individual-record level - one row per split species value, same convention as `species_study.ipynb`.

1. Resolve SEAD ids for the manually-completed species list.
2. Split/melt `c14_master_v08.xlsx`'s `species` column, map each split value through the manual
   correction, then attach the resolved SEAD taxonomy/ids as new columns.

In [1]:
import re
from pathlib import Path

import pandas as pd

pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

OUTPUT_DIR = Path('../output/species')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# The transformed c14 dataset (step 2's output) is a dataset export, not a species-resolution
# artifact, so it gets its own folder rather than living alongside the species/taxonomy CSVs.
MOD_DATASET_DIR = Path('../output/mod_dataset')
MOD_DATASET_DIR.mkdir(parents=True, exist_ok=True)

# Matches the manual mapping revision the resolved species list (data/manual_species_resolved_test.csv)
# was built from - only used here to rebuild the species_split -> manual_species correction table.
DATA_VERSION = 'v4'
MANUAL_MAPPING_PATH = Path(f'../data/species_split_counts_in_original_manual_{DATA_VERSION}.csv')

def versioned(filename):
    # e.g. 'manual_species_count.csv' -> 'manual_species_count_v4.csv'
    stem, suffix = filename.rsplit('.', 1)
    return f'{stem}_{DATA_VERSION}.{suffix}'

def next_available_path(filename, dir_path=OUTPUT_DIR):
    # Never silently overwrite a previous run's output - if the versioned filename is already
    # taken, keep bumping a numeric suffix (_2, _3, ...) until a free one is found.
    versioned_name = versioned(filename)
    candidate = dir_path / versioned_name
    if not candidate.exists():
        return candidate
    stem, suffix = versioned_name.rsplit('.', 1)
    n = 2
    while (dir_path / f'{stem}_{n}.{suffix}').exists():
        n += 1
    return dir_path / f'{stem}_{n}.{suffix}'

## Connect to sead_staging and load the taxonomy lookups

In [2]:
import os

from dotenv import load_dotenv
from sqlalchemy import create_engine

load_dotenv('../.env')

DB_HOST = os.environ["DB_HOST"]
DB_PORT = os.environ["DB_PORT"]
DB_NAME = os.environ["DB_NAME"]
DB_USER = os.environ["DB_USER"]
DB_PASSWORD = os.environ["DB_PASSWORD"]

engine = create_engine(f"postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}")

In [3]:
# LEFT JOIN straight against the public tables (not view_taxa_alphabetically, which silently
# drops/omits rows it can't fully resolve). taxa_master/common_names_all/sv_common are all derived
# from this single source so they can't drift out of sync with each other.
taxa_raw = pd.read_sql(
    """
    select
        m.taxon_id, m.genus_id, m.species, m.author_id,
        g.genus_name, f.family_name, o.order_name,
        c.taxon_common_name_id, c.common_name, c.language_id
    from public.tbl_taxa_tree_master m
    left join public.tbl_taxa_tree_genera g on m.genus_id = g.genus_id
    left join public.tbl_taxa_tree_families f on g.family_id = f.family_id
    left join public.tbl_taxa_tree_orders o on f.order_id = o.order_id
    left join public.tbl_taxa_common_names c on m.taxon_id = c.taxon_id
    """,
    engine,
)

genera = pd.read_sql(
    'select lower(genus_name) as genus_lc, genus_name, genus_id, family_id from public.tbl_taxa_tree_genera', engine
)
families = pd.read_sql(
    'select lower(family_name) as family_lc, family_name, family_id, order_id from public.tbl_taxa_tree_families', engine
)
orders_lookup = pd.read_sql(
    'select lower(order_name) as order_lc, order_name, order_id from public.tbl_taxa_tree_orders', engine
)

# One row per taxon_id (author_id kept - SEAD can have several taxon_id rows for the exact same
# genus+species text, differing only in which taxonomic authority/author_id they're attributed to).
taxa_master = taxa_raw.drop_duplicates('taxon_id')[
    ['taxon_id', 'genus_id', 'species', 'author_id', 'genus_name', 'family_name', 'order_name']
].reset_index(drop=True)

common_names_all = (
    taxa_raw.dropna(subset=['taxon_common_name_id'])[
        ['taxon_common_name_id', 'common_name', 'taxon_id', 'language_id']
    ]
    .drop_duplicates()
    .astype({'taxon_common_name_id': 'int64', 'language_id': 'int64'})
    .reset_index(drop=True)
)

sv_common = taxa_raw[(taxa_raw['language_id'] == 2) & taxa_raw['common_name'].notna()].copy()
sv_common['common_name_lc'] = sv_common['common_name'].str.lower()
common_map = sv_common.drop_duplicates('common_name_lc').set_index('common_name_lc')

genus_hierarchy = (
    genera.merge(families[['family_id', 'family_name', 'order_id']], on='family_id', how='left')
          .merge(orders_lookup[['order_id', 'order_name']], on='order_id', how='left')
          .drop_duplicates('genus_lc').set_index('genus_lc')
)
family_hierarchy = (
    families.merge(orders_lookup[['order_id', 'order_name']], on='order_id', how='left')
             .drop_duplicates('family_lc').set_index('family_lc')
)
order_hierarchy = orders_lookup.drop_duplicates('order_lc').set_index('order_lc')

print(f'{len(sv_common)} Swedish common names, {len(genus_hierarchy)} genera, '
      f'{len(family_hierarchy)} families, {len(order_hierarchy)} orders, {len(taxa_master)} taxa loaded '
      f'({taxa_master["author_id"].notna().sum()} taxa have an author_id)')

4272 Swedish common names, 5212 genera, 541 families, 57 orders, 23981 taxa loaded (20128 taxa have an author_id)


## Resolver functions

Same `resolve_order`/`resolve_family`/`resolve_genus`/`resolve_taxon`/`resolve_common_name` as
`species_split_for_study_etl.ipynb` step 5: reuse an existing SEAD row whenever the Latin name
already matches one (case-insensitive), otherwise propose a new one - `"<parent> indet"` when the
rank itself is unknown (the one existing `Cyperaceae indet` precedent). `resolve_taxon` only treats
an existing taxon as reusable when its `author_id` is NULL; if every existing match for that exact
species text has an author_id, a fresh taxon (`author_id` NULL) is proposed instead.

In [4]:
next_order_id = int(orders_lookup['order_id'].max()) + 1
next_family_id = int(families['family_id'].max()) + 1
next_genus_id = int(genera['genus_id'].max()) + 1
next_taxon_id = int(taxa_master['taxon_id'].max()) + 1
next_common_name_id = int(common_names_all['taxon_common_name_id'].max()) + 1

LANGUAGE_NAMES = {1: 'English', 2: 'Swedish'}

proposed_orders = {}         # order_name_lc -> order_id
proposed_families = {}       # family_name_lc -> family_id
proposed_genera = {}         # genus_name_lc -> genus_id
proposed_taxa = {}           # (genus_id, species_lc) -> taxon_id
proposed_common_names = {}   # (taxon_id, language_id, common_name_lc) -> taxon_common_name_id
new_records = []             # rows for new_sead_records_manual_resolution.csv
CURRENT_MANUAL_SPECIES = None

def resolve_order(order_name):
    """Returns (order_id, is_new, name_used)."""
    if order_name is None:
        return None, False, None
    lc = order_name.lower()
    if lc in order_hierarchy.index:
        return int(order_hierarchy.loc[lc, 'order_id']), False, order_name
    if lc in proposed_orders:
        return proposed_orders[lc], True, order_name
    global next_order_id
    order_id = next_order_id
    next_order_id += 1
    proposed_orders[lc] = order_id
    new_records.append(dict(table='tbl_taxa_tree_orders', id_column='order_id', id=order_id,
                             name=order_name, language_id=None, author_id=None,
                             parent_table=None, parent_id=None, created_for=CURRENT_MANUAL_SPECIES))
    return order_id, True, order_name

def resolve_family(family_name, order_id, order_name_for_placeholder):
    """order_id must already be resolved by the caller. family_name=None -> propose a
    '<order> indet' placeholder family under order_id (or generic 'Indet' if the order itself is
    unknown too). Returns (family_id, is_new, name_used)."""
    if family_name is None:
        family_name = f'{order_name_for_placeholder} indet' if order_name_for_placeholder else 'Indet'
    lc = family_name.lower()
    if lc in family_hierarchy.index:
        return int(family_hierarchy.loc[lc, 'family_id']), False, family_name
    if lc in proposed_families:
        return proposed_families[lc], True, family_name
    global next_family_id
    family_id = next_family_id
    next_family_id += 1
    proposed_families[lc] = family_id
    new_records.append(dict(table='tbl_taxa_tree_families', id_column='family_id', id=family_id,
                             name=family_name, language_id=None, author_id=None,
                             parent_table='tbl_taxa_tree_orders', parent_id=order_id,
                             created_for=CURRENT_MANUAL_SPECIES))
    return family_id, True, family_name

def resolve_genus(genus_name, family_id, family_name_for_placeholder):
    """family_id must already be resolved by the caller. genus_name=None -> propose a
    '<family> indet' placeholder genus under family_id. Returns (genus_id, is_new, name_used)."""
    if genus_name is None:
        genus_name = f'{family_name_for_placeholder} indet' if family_name_for_placeholder else 'Indet'
    lc = genus_name.lower()
    if lc in genus_hierarchy.index:
        return int(genus_hierarchy.loc[lc, 'genus_id']), False, genus_name
    if lc in proposed_genera:
        return proposed_genera[lc], True, genus_name
    global next_genus_id
    genus_id = next_genus_id
    next_genus_id += 1
    proposed_genera[lc] = genus_id
    new_records.append(dict(table='tbl_taxa_tree_genera', id_column='genus_id', id=genus_id,
                             name=genus_name, language_id=None, author_id=None,
                             parent_table='tbl_taxa_tree_families', parent_id=family_id,
                             created_for=CURRENT_MANUAL_SPECIES))
    return genus_id, True, genus_name

def resolve_taxon(genus_id, species_epithet):
    """species_epithet=None means we only know the genus - look for/propose that genus's
    indeterminate-species placeholder (SEAD uses both 'indet.' and 'sp.' for this - check both).
    Only an existing taxon with author_id IS NULL is safe to reuse outright.
    Returns (taxon_id, is_new, species_text_used, blocked_by_author_id)."""
    target_lc = (species_epithet or 'indet.').lower().rstrip('.')
    genus_taxa = taxa_master[taxa_master['genus_id'] == genus_id]

    if species_epithet is not None:
        candidates = genus_taxa[genus_taxa['species'].str.lower().str.rstrip('.') == target_lc]
    else:
        candidates = genus_taxa[
            (genus_taxa['species'].str.lower().str.rstrip('.') == target_lc)
            | genus_taxa['species'].str.lower().str.startswith(('indet', 'sp.', 'spp.'))
        ]

    usable = candidates[candidates['author_id'].isna()]
    blocked_by_author_id = len(candidates) > 0 and usable.empty

    if len(usable):
        row0 = usable.iloc[0]
        return int(row0['taxon_id']), False, row0['species'], blocked_by_author_id

    key = (genus_id, target_lc)
    species_text = species_epithet or 'indet.'
    if key in proposed_taxa:
        return proposed_taxa[key], True, species_text, blocked_by_author_id
    global next_taxon_id
    taxon_id = next_taxon_id
    next_taxon_id += 1
    proposed_taxa[key] = taxon_id
    new_records.append(dict(table='tbl_taxa_tree_master', id_column='taxon_id', id=taxon_id,
                             name=species_text, language_id=None, author_id=None,
                             parent_table='tbl_taxa_tree_genera', parent_id=genus_id,
                             created_for=CURRENT_MANUAL_SPECIES))
    return taxon_id, True, species_text, blocked_by_author_id

def resolve_common_name(taxon_id, common_name_text, language_id, taxon_is_new):
    common_name_lc = common_name_text.rstrip('?').strip().lower()
    if not taxon_is_new:
        existing = common_names_all[
            (common_names_all['taxon_id'] == taxon_id) & (common_names_all['language_id'] == language_id)
        ]
        if len(existing):
            return int(existing.iloc[0]['taxon_common_name_id']), False
    key = (taxon_id, language_id, common_name_lc)
    if key in proposed_common_names:
        return proposed_common_names[key], True
    global next_common_name_id
    common_name_id = next_common_name_id
    next_common_name_id += 1
    proposed_common_names[key] = common_name_id
    new_records.append(dict(table='tbl_taxa_common_names', id_column='taxon_common_name_id', id=common_name_id,
                             name=common_name_text, language_id=language_id, author_id=None,
                             parent_table='tbl_taxa_tree_master', parent_id=taxon_id,
                             created_for=CURRENT_MANUAL_SPECIES))
    return common_name_id, True

## 1. Resolve real SEAD ids for the manually-completed species list

`data/manual_species_resolved_test.csv` is a hand-completed version of the automated pass's
output: every row now has `resolved_order`/`resolved_family`/`resolved_genus`/`resolved_species`/
`common_name_text` filled in, including rows that used to be blank (e.g. genus-level `Indet.`
placeholders for values with no specific species - including the row for records with no species
value at all, `manual_species` itself blank). Its own `taxon_id`/`resolved_genus_id`/
`common_name_id` columns aren't trusted as-is (some are stale synthetic ids from an earlier
automated run, no longer meaningful once the row set changed); every id below is recomputed from
scratch purely from the resolved *names*.

In [5]:
RESOLUTION_COLUMNS = [
    'manual_species', 'contributing_species_split', 'resolved_order', 'resolved_order_is_new',
    'resolved_family', 'resolved_family_is_new', 'resolved_genus', 'resolved_genus_id',
    'resolved_genus_is_new', 'resolved_species', 'resolved_scientific_name', 'taxon_id',
    'taxon_id_is_new', 'matched_existing_taxon', 'blocked_by_existing_author_id',
    'common_name_id', 'common_name_id_is_new', 'common_name_text', 'common_name_language',
]

manually_resolved_df = pd.read_csv(
    '../data/manual_species_resolved_test.csv', keep_default_na=False, na_values=[''],
)[RESOLUTION_COLUMNS]
print(f'{len(manually_resolved_df)} rows loaded, restricted to the resolution columns')
manually_resolved_df.head()

212 rows loaded, restricted to the resolution columns


,manual_species,contributing_species_split,resolved_order,resolved_order_is_new,resolved_family,resolved_family_is_new,resolved_genus,resolved_genus_id,resolved_genus_is_new,resolved_species,resolved_scientific_name,taxon_id,taxon_id_is_new,matched_existing_taxon,blocked_by_existing_author_id,common_name_id,common_name_id_is_new,common_name_text,common_name_language
0,al,"al, albark, alknopp, alkottar, alkotte",Fagales,False,Betulaceae,False,Alnus,263.0,False,sp.,NaN,18087.0,False,False,False,NaN,True,al,2.0
1,tall,"kottefjäll tall, kottefjäll. tall, tall, tallb...",Pinales,False,Pinaceae,False,Pinus,764.0,False,sylvestris var sylvestris,Pinus sylvestris var sylvestris,3613.0,False,True,False,2797.0,False,tall,2.0
2,korn,"korn, kornhalm, skalkorn",Poales,False,Poaceae,False,Hordeum,830.0,False,vulgare,NaN,18010.0,False,False,False,NaN,True,korn,2.0
3,björk,"bjrök, björk, björk bulk, björkl, björknäver",Fagales,False,Betulaceae,False,Betula,264.0,False,sp.,Betula sp.,18086.0,False,True,False,4273.0,True,björk,2.0
4,ek,"ek, ek bulk, ekbark, ekl, ekollon",Fagales,False,Fagaceae,False,Quercus,551.0,False,robur,Quercus robur,47010.0,True,False,True,4274.0,True,ek,2.0


In [6]:
def resolve_manual_row(row):
    global CURRENT_MANUAL_SPECIES
    CURRENT_MANUAL_SPECIES = row['manual_species']

    order = row['resolved_order'] if pd.notna(row['resolved_order']) else None
    family = row['resolved_family'] if pd.notna(row['resolved_family']) else None
    genus = row['resolved_genus'] if pd.notna(row['resolved_genus']) else None
    species_epithet = row['resolved_species'] if pd.notna(row['resolved_species']) else None

    order_id, order_is_new = None, False
    family_id, family_is_new = None, False
    genus_id, genus_is_new = None, False
    taxon_id, taxon_is_new, blocked_by_author_id = None, False, False

    if order or family or genus:
        order_id, order_is_new, order_name_used = resolve_order(order)
        family_id, family_is_new, family_name_used = resolve_family(family, order_id, order)
        genus_id, genus_is_new, genus_name_used = resolve_genus(genus, family_id, family_name_used)
        taxon_id, taxon_is_new, species_used, blocked_by_author_id = resolve_taxon(genus_id, species_epithet)

    common_name_id, common_name_is_new = None, False
    common_name_text = row['common_name_text'] if pd.notna(row['common_name_text']) else None
    common_name_language = row['common_name_language'] if pd.notna(row['common_name_language']) else None
    if taxon_id is not None and common_name_text is not None and common_name_language is not None:
        common_name_id, common_name_is_new = resolve_common_name(
            taxon_id, common_name_text, int(common_name_language), taxon_is_new
        )

    return pd.Series(dict(
        resolved_order_id=order_id, resolved_order_is_new=bool(order_is_new),
        resolved_family_id=family_id, resolved_family_is_new=bool(family_is_new),
        resolved_genus_id=genus_id, resolved_genus_is_new=bool(genus_is_new),
        taxon_id=taxon_id, taxon_id_is_new=bool(taxon_is_new),
        matched_existing_taxon=(taxon_id is not None and not taxon_is_new),
        blocked_by_existing_author_id=bool(blocked_by_author_id),
        common_name_id=common_name_id, common_name_id_is_new=bool(common_name_is_new),
    ))

resolution = manually_resolved_df.apply(resolve_manual_row, axis=1)
manually_resolved_final = pd.concat(
    [
        manually_resolved_df.drop(columns=[
            'resolved_genus_id', 'resolved_genus_is_new', 'taxon_id', 'taxon_id_is_new',
            'matched_existing_taxon', 'blocked_by_existing_author_id', 'common_name_id',
            'common_name_id_is_new',
        ]),
        resolution,
    ],
    axis=1,
)
print(f"{manually_resolved_final['taxon_id'].notna().sum()} of {len(manually_resolved_final)} rows "
      f"resolved to a taxon_id ({int(manually_resolved_final['taxon_id_is_new'].sum())} newly proposed)")
print(f"{int(manually_resolved_final['blocked_by_existing_author_id'].sum())} rows had a matching "
      f"species blocked by an existing author_id")
print(f'{len(new_records)} new SEAD records proposed for this list')
manually_resolved_final.head()

212 of 212 rows resolved to a taxon_id (139 newly proposed)
45 rows had a matching species blocked by an existing author_id
306 new SEAD records proposed for this list


,manual_species,contributing_species_split,resolved_order,resolved_order_is_new,resolved_family,resolved_family_is_new,resolved_genus,resolved_species,resolved_scientific_name,common_name_text,common_name_language,resolved_order_id,resolved_order_is_new,resolved_family_id,resolved_family_is_new,resolved_genus_id,resolved_genus_is_new,taxon_id,taxon_id_is_new,matched_existing_taxon,blocked_by_existing_author_id,common_name_id,common_name_id_is_new
0,al,"al, albark, alknopp, alkottar, alkotte",Fagales,False,Betulaceae,False,Alnus,sp.,NaN,al,2.0,19.0,False,32,False,263,False,18087,False,True,False,4273.0,True
1,tall,"kottefjäll tall, kottefjäll. tall, tall, tallb...",Pinales,False,Pinaceae,False,Pinus,sylvestris var sylvestris,Pinus sylvestris var sylvestris,tall,2.0,33.0,False,137,False,764,False,3613,False,True,False,2797.0,False
2,korn,"korn, kornhalm, skalkorn",Poales,False,Poaceae,False,Hordeum,vulgare,NaN,korn,2.0,34.0,False,141,False,830,False,18010,False,True,False,4274.0,True
3,björk,"bjrök, björk, björk bulk, björkl, björknäver",Fagales,False,Betulaceae,False,Betula,sp.,Betula sp.,björk,2.0,19.0,False,32,False,264,False,18086,False,True,False,4275.0,True
4,ek,"ek, ek bulk, ekbark, ekl, ekollon",Fagales,False,Fagaceae,False,Quercus,robur,Quercus robur,ek,2.0,19.0,False,76,False,551,False,47010,True,False,True,4276.0,True


In [7]:
RESOLVED_IDS_PATH = next_available_path('manual_species_resolved_with_ids.csv')
manually_resolved_final.to_csv(RESOLVED_IDS_PATH, index=False)

new_sead_records_manual = pd.DataFrame(
    new_records,
    columns=['table', 'id_column', 'id', 'name', 'language_id', 'author_id', 'parent_table', 'parent_id', 'created_for'],
)
NEW_SEAD_RECORDS_MANUAL_PATH = next_available_path('new_sead_records_manual_resolution.csv')
new_sead_records_manual.to_csv(NEW_SEAD_RECORDS_MANUAL_PATH, index=False)

print(f'Saved {len(manually_resolved_final)} rows to {RESOLVED_IDS_PATH.name}')
print(f'Saved {len(new_sead_records_manual)} proposed new records to {NEW_SEAD_RECORDS_MANUAL_PATH.name}')
new_sead_records_manual['table'].value_counts()

Saved 212 rows to manual_species_resolved_with_ids_v4_3.csv
Saved 306 proposed new records to new_sead_records_manual_resolution_v4_3.csv


table
tbl_taxa_common_names     139
tbl_taxa_tree_master       92
tbl_taxa_tree_genera       35
tbl_taxa_tree_families     24
tbl_taxa_tree_orders       16
Name: count, dtype: int64

## 2. Map the resolved taxonomy back onto `c14_master_v08.xlsx`, one species per row

Same split + melt as `species_study.ipynb` (split the raw `species` column on `,`/`/`, one atomic
`species_split` value per row, `fid` preserved), then the `species_split` -> `manual_species`
correction (rebuilt fresh from the manual mapping CSV here, comma-split the same way step 1 of
`species_split_for_study_etl.ipynb` does), then `manually_resolved_final` (step 1 above) attaches
the SEAD taxonomy/ids for each resolved term.

In [8]:
manual_map_raw = pd.read_csv(
    MANUAL_MAPPING_PATH, keep_default_na=False, na_values=[''],
)[['species_split', 'manual_species']]

def split_manual_species(value):
    if pd.isna(value):
        return [pd.NA]
    return [part.strip() for part in value.split(',')]

manual_species_split_map = pd.DataFrame([
    {'species_split': row.species_split, 'manual_species': part}
    for row in manual_map_raw.itertuples(index=False)
    for part in split_manual_species(row.manual_species)
])
print(f'{len(manual_map_raw)} species_split -> manual_species rows -> '
      f'{len(manual_species_split_map)} after comma-splitting manual_species')

411 species_split -> manual_species rows -> 415 after comma-splitting manual_species


In [9]:
c14_df = pd.read_excel('../data/c14_master_v08.xlsx')
c14_df['species'] = c14_df['species'].str.lower()

noise_pattern = re.compile(r'[0-9?]')

def clean_text(value):
    if pd.isna(value) or value == '':
        return pd.NA
    cleaned = noise_pattern.sub('', value).strip()
    return cleaned if cleaned else pd.NA

species_parts = c14_df['species'].str.split(r'[,/]', expand=True)
species_parts.columns = [f'species_{i + 1}' for i in range(species_parts.shape[1])]
species_parts = species_parts.apply(lambda col: col.map(clean_text))
c14_df = c14_df.join(species_parts)

species_cols = [c for c in c14_df.columns if c.startswith('species_')]
species_split = c14_df[species_cols].stack().dropna().droplevel(1).rename('species_split')
c14_melted = c14_df.drop(columns=species_cols).join(species_split).reset_index(drop=True)
print(f'{len(c14_df)} original rows -> {len(c14_melted)} rows after splitting/melting species')

30301 original rows -> 30832 rows after splitting/melting species


In [10]:
c14_with_manual = c14_melted.merge(manual_species_split_map, on='species_split', how='left')
print(f"{c14_with_manual['manual_species'].notna().sum()} of {len(c14_with_manual)} melted rows "
      f"matched a manual_species correction ({len(c14_melted)} rows before this join, "
      f"{len(c14_with_manual)} after - the difference is from manual_species values that are "
      f"themselves comma-split into multiple candidates)")
c14_with_manual[['fid', 'species', 'species_split', 'manual_species']].head(10)

18774 of 30838 melted rows matched a manual_species correction (30832 rows before this join, 30838 after - the difference is from manual_species values that are themselves comma-split into multiple candidates)


,fid,species,species_split,manual_species
0,1,lind,lind,lind
1,2,NaN,NaN,NaN
2,3,NaN,NaN,NaN
3,4,NaN,NaN,NaN
4,5,NaN,NaN,NaN
5,6,NaN,NaN,NaN
6,7,NaN,NaN,NaN
7,8,NaN,NaN,NaN
8,9,NaN,NaN,NaN
9,10,NaN,NaN,NaN


In [11]:
SEAD_COLUMN_MAP = {
    'common_name_text': 'common_name',
    'resolved_species': 'sead_species',
    'resolved_genus': 'sead_genus',
    'resolved_family': 'sead_family',
    'resolved_order': 'sead_order',
    'taxon_id': 'sead_species_id',
    'resolved_genus_id': 'sead_genus_id',
    'resolved_family_id': 'sead_family_id',
    'resolved_order_id': 'sead_order_id',
    'common_name_id': 'sead_common_name_id',
    'common_name_language': 'sead_common_name_language_id',
}

resolved_for_join = manually_resolved_final[['manual_species'] + list(SEAD_COLUMN_MAP)].rename(
    columns=SEAD_COLUMN_MAP
)

c14_with_sead = c14_with_manual.merge(resolved_for_join, on='manual_species', how='left')
c14_with_sead = c14_with_sead.rename(columns={'manual_species': 'species_manually_assigned'})

print(f"{c14_with_sead['sead_species_id'].notna().sum()} of {len(c14_with_sead)} rows have a "
      f"sead_species_id attached")
c14_with_sead[
    ['fid', 'species', 'species_split', 'species_manually_assigned', 'common_name', 'sead_genus',
     'sead_family', 'sead_order', 'sead_species_id', 'sead_genus_id', 'sead_family_id',
     'sead_order_id', 'sead_common_name_id', 'sead_common_name_language_id']
].head(10)

30838 of 30838 rows have a sead_species_id attached


,fid,species,species_split,species_manually_assigned,common_name,sead_genus,sead_family,sead_order,sead_species_id,sead_genus_id,sead_family_id,sead_order_id,sead_common_name_id,sead_common_name_language_id
0,1,lind,lind,lind,lind,Tilia,Tiliaceae,Malvales,47020,1052,176,27.0,4287.0,2.0
1,2,NaN,NaN,NaN,NaN,Indet.,Indet.,Indet.,47014,16753,1994,142.0,NaN,NaN
2,3,NaN,NaN,NaN,NaN,Indet.,Indet.,Indet.,47014,16753,1994,142.0,NaN,NaN
3,4,NaN,NaN,NaN,NaN,Indet.,Indet.,Indet.,47014,16753,1994,142.0,NaN,NaN
4,5,NaN,NaN,NaN,NaN,Indet.,Indet.,Indet.,47014,16753,1994,142.0,NaN,NaN
5,6,NaN,NaN,NaN,NaN,Indet.,Indet.,Indet.,47014,16753,1994,142.0,NaN,NaN
6,7,NaN,NaN,NaN,NaN,Indet.,Indet.,Indet.,47014,16753,1994,142.0,NaN,NaN
7,8,NaN,NaN,NaN,NaN,Indet.,Indet.,Indet.,47014,16753,1994,142.0,NaN,NaN
8,9,NaN,NaN,NaN,NaN,Indet.,Indet.,Indet.,47014,16753,1994,142.0,NaN,NaN
9,10,NaN,NaN,NaN,NaN,Indet.,Indet.,Indet.,47014,16753,1994,142.0,NaN,NaN


In [12]:
C14_WITH_SEAD_PATH = next_available_path('c14_master_v08_with_sead_taxonomy.csv', dir_path=MOD_DATASET_DIR)
c14_with_sead.to_csv(C14_WITH_SEAD_PATH, index=False)
print(f'Saved {len(c14_with_sead)} rows to {C14_WITH_SEAD_PATH}')

Saved 30838 rows to ..\output\mod_dataset\c14_master_v08_with_sead_taxonomy_v4_3.csv
